# Arctic Ocean

Arctic Ocean hydrography

In [ ]:
# parameters for the cmorized data
caseid          = ''
input_root      = ''
work_root       = ''
www_root        = ''
year_start      = []
year_end        = []

In [ ]:
import yaml
import xarray as xr
import sys
import os
import numpy as np
import matplotlib.pyplot as plt
import cartopy.crs as ccrs
import cartopy.feature as cfeature
from pathlib import Path
import matplotlib as mpl
import cartopy.crs as ccrs
import matplotlib.path as mpath
import xesmf as xe
from pathlib import Path
import matplotlib.colors as mcolors
from scipy.spatial import cKDTree
with open("../settings.yml", "r") as f:
    cfg = yaml.safe_load(f)
sys.path.append(cfg["blomster"]["source_root"])
from utils.plots import plot_map2d
from utils import get_grid
from utils.analysis import mean_at_depth,load_ocean_grid,deep_ocean_mask,shared_quantile_levels,symmetric_quantile_levels


In [ ]:
caseid = cfg['experiment']['caseid']
input_root = cfg['experiment']['input_root']
work_root = cfg['experiment']['work_root']
www_root = cfg['experiment']['www_root']
year_start = cfg['experiment']['year_start']
year_end = cfg['experiment']['year_end']
years = range(year_start, year_end+1)
grid_file = cfg['grids']['tnx1v4']['grid_file']
phc_file = cfg['datasets']['PHC3.0']['ANNUAL_file']

time_coder = xr.coders.CFDatetimeCoder(use_cftime=True)
hist_dir = Path(f"{input_root}/{caseid}/ocn/hist")
files = [hist_dir / f"{caseid}.blom.hy.{yr}.nc" for yr in years]

## Hovmoller diagrams

### 1.values change with time

In [ ]:
max_depth = 1500
with xr.open_dataset(grid_file) as grid:
    arctic_mask = ((grid["pmask"] > 0)& (grid["plat"] >= 65)& (grid["pdepth"] > 500))
    weights = (grid["parea"].where(arctic_mask, 0).fillna(0).load())
    
temp_profiles = []
salt_profiles = []
for i, file in enumerate(files, 1):
    with xr.open_dataset(file, decode_times=time_coder) as ds:
        temp_one = (ds["templvl"].sel(depth=slice(0, 4000)).weighted(weights).mean(("y", "x"), skipna=True).load())
        salt_one = (ds["salnlvl"].sel(depth=slice(0, 4000)).weighted(weights).mean(("y", "x"), skipna=True).load())
    temp_profiles.append(temp_one)
    salt_profiles.append(salt_one)
temp_time_depth = xr.concat(temp_profiles, dim="time")
salt_time_depth = xr.concat(salt_profiles, dim="time")

temp_time_levels = shared_quantile_levels([temp_time_depth],nlevels=21)
salt_time_levels = shared_quantile_levels([salt_time_depth],nlevels=21)

In [ ]:
plot_years = temp_time_depth["time"].dt.year.values
fig, ax = plt.subplots(figsize=(9, 6))
mesh = ax.pcolormesh(
    plot_years,
    temp_time_depth["depth"] / 1000,
    temp_time_depth.transpose("depth", "time"),
    cmap="plasma",
    vmin=temp_time_levels[0],
    vmax=temp_time_levels[-1],
    shading="auto",
)

colorbar = fig.colorbar(mesh, ax=ax, pad=0.02)
colorbar.set_label("Temperature [°C]")

ax.set_xlabel("Model year",fontsize=12)
ax.set_ylabel("Depth [km]",fontsize=12)
ax.set_title("Arctic Ocean temperature",fontsize=14)
ax.set_ylim(max_depth / 1000, 0)

fig.suptitle(f"{caseid}\nArea-weighted mean north of 65°N",fontsize=12,y=0.98)

fig.tight_layout(rect=[0, 0, 1, 0.97])
plt.show()

In [ ]:
fig, ax = plt.subplots(figsize=(9, 6))
mesh = ax.pcolormesh(
    plot_years,
    salt_time_depth["depth"] / 1000,
    salt_time_depth.transpose("depth", "time"),
    cmap="plasma",
    vmin=salt_time_levels[0],
    vmax=salt_time_levels[-1],
    shading="auto",
)

colorbar = fig.colorbar(mesh, ax=ax, pad=0.02)
colorbar.set_label("Salnility [g/kg]")

ax.set_xlabel("Model year",fontsize=12)
ax.set_ylabel("Depth [km]",fontsize=12)
ax.set_title("Arctic Ocean Salnility",fontsize=14)
ax.set_ylim(max_depth / 1000, 0)
fig.suptitle(f"{caseid}\nArea-weighted mean north of 65°N",fontsize=12,y=0.98)
fig.tight_layout(rect=[0, 0, 1, 0.97])
plt.show()

### 2. anomaly change with time

In [ ]:
clim_temp = temp_time_depth.isel(time=slice(0, 10)).mean("time", skipna=True)
clim_salt = salt_time_depth.isel(time=slice(0, 10)).mean("time", skipna=True)

temp_anomaly = temp_time_depth- clim_temp
salt_anomaly = salt_time_depth- clim_salt
temp_ano_levels = symmetric_quantile_levels(temp_anomaly,nlevels=21)
salt_ano_levels = symmetric_quantile_levels(salt_anomaly,nlevels=21)
fig, ax = plt.subplots(figsize=(9, 6))
mesh = ax.pcolormesh(
    plot_years,
    temp_anomaly["depth"] / 1000,
    temp_anomaly.transpose("depth", "time"),
    cmap="RdBu_r",
    vmin=temp_ano_levels[0],
    vmax=temp_ano_levels[-1],
    shading="auto",
)

colorbar = fig.colorbar(mesh, ax=ax, pad=0.02)
colorbar.set_label("Temperature [°C]")

ax.set_xlabel("Model year",fontsize=12)
ax.set_ylabel("Depth [km]",fontsize=12)
ax.set_title("Arctic Ocean temperature anomaly",fontsize=14)
ax.set_ylim(max_depth / 1000, 0)

fig.suptitle(f"{caseid}\nArea-weighted mean north of 65°N",fontsize=12,y=0.98)

fig.tight_layout(rect=[0, 0, 1, 0.97])
plt.show()

In [ ]:
fig, ax = plt.subplots(figsize=(9, 6))
mesh = ax.pcolormesh(
    plot_years,
    salt_anomaly["depth"] / 1000,
    salt_anomaly.transpose("depth", "time"),
    cmap="RdBu_r",
    vmin=salt_ano_levels[0],
    vmax=salt_ano_levels[-1],
    shading="auto",
)

colorbar = fig.colorbar(mesh, ax=ax, pad=0.02)
colorbar.set_label("Salnility [g/kg]")

ax.set_xlabel("Model year",fontsize=12)
ax.set_ylabel("Depth [km]",fontsize=12)
ax.set_title("Arctic Ocean sanility anomaly",fontsize=14)
ax.set_ylim(max_depth / 1000, 0)

fig.suptitle(f"{caseid}\nArea-weighted mean north of 65°N",fontsize=12,y=0.98)

fig.tight_layout(rect=[0, 0, 1, 0.97])
plt.show()

## Vertical 
1. temp profile with observation
2. salt profile with observation

In [ ]:
with xr.open_dataset(grid_file) as grid:
    ydim, xdim = grid["plon"].dims
    target_grid = xr.Dataset({"lon": ((ydim, xdim),np.mod(grid["plon"].values, 360)),"lat": ((ydim, xdim),grid["plat"].values)})
    arctic_mask = ((grid["pmask"] > 0) & (grid["plat"] >= 65) & (grid["pdepth"] > 500))
    weights = (grid["parea"].where(arctic_mask, 0).fillna(0).load())

temp_profiles = []
salt_profiles = []

for i, file in enumerate(files, 1):
    with xr.open_dataset(file, decode_times=time_coder) as ds:
        temp_one = (ds["templvl"].sel(depth=slice(0, 4000)).weighted(weights).mean(("y", "x"), skipna=True).load())
        salt_one = (ds["salnlvl"].sel(depth=slice(0, 4000)).weighted(weights).mean(("y", "x"), skipna=True).load())
    temp_profiles.append(temp_one)
    salt_profiles.append(salt_one)

temp_profile = (xr.concat(temp_profiles, dim="time").mean("time", skipna=True))
salt_profile = (xr.concat(salt_profiles, dim="time").mean("time", skipna=True))


with xr.open_dataset(phc_file,decode_times=time_coder,) as phc:
    phc_grid = xr.Dataset(coords={"lon": phc["lon"],"lat": phc["lat"]})
    regridder = xe.Regridder(phc_grid,target_grid,method="bilinear",periodic=True)
    phc_temp_3d_on_model = regridder(phc["temp"].sel(depth=slice(0, 4000)),skipna=True,na_thres=1.0).load()
    phc_salt_3d_on_model = regridder(phc["salt"].sel(depth=slice(0, 4000)),skipna=True,na_thres=1.0).load()

phc_temp_profile = (phc_temp_3d_on_model.weighted(weights).mean(("y", "x"), skipna=True))
phc_salt_profile = (phc_salt_3d_on_model.weighted(weights).mean(("y", "x"), skipna=True))


In [ ]:
fig, ax = plt.subplots(figsize=(6, 6.5))
ax.plot(temp_profile,temp_profile["depth"] / 1000,color="tab:red",linewidth=2.2,label="NorESM3")
ax.plot(phc_temp_profile,phc_temp_profile["depth"] / 1000,color="black",linestyle="-",linewidth=2,label="PHC3.0")
ax.set_xlabel("Temperature [°C]",fontsize=12)
ax.set_ylabel("Depth [km]",fontsize=12)
ax.set_title("Arctic Ocean temperature profile",fontsize=14)
ax.set_ylim(4, 0)
ax.grid(True, alpha=0.3)
ax.legend()
fig.suptitle(f"{caseid}\nYears {year_start}–{year_end}",fontsize=12,y=0.98)
fig.tight_layout(rect=[0, 0, 1, 0.95])
plt.show()

In [ ]:
fig, ax = plt.subplots(figsize=(6, 6.5))
ax.plot(salt_profile,salt_profile["depth"] / 1000,color="tab:blue",linewidth=2.2,label="NorESM3",)
ax.plot(phc_salt_profile,phc_salt_profile["depth"] / 1000,color="black",linestyle="-",linewidth=2,label="PHC3.0")
ax.set_xlabel("Salinity [g kg$^{-1}$]",fontsize=12)
ax.set_ylabel("Depth [km]",fontsize=12)
ax.set_title("Arctic Ocean salinity profile",fontsize=14)
ax.set_ylim(4, 0)
ax.grid(True, alpha=0.3)
ax.legend()
fig.suptitle(f"{caseid}\nYears {year_start}–{year_end}",fontsize=12,y=0.98)
fig.tight_layout(rect=[0, 0, 1, 0.95])
plt.show()

In [ ]:
## define Lomonosov Ridge:
ridge_lat = np.array([80.8, 82.5, 84.0, 86.0, 87.5, 88.7,89.2, 88.9, 88.2, 87.1, 85.0, 83.0])
ridge_lon = np.array([140, 145, 150, 155, 165, 175,-170, -150, -120, -90, -70, -60])
def polar_xy(lon, lat):
    """
    Simple north-pole azimuthal projection.
    Only used for polygon point-in-polygon calculations.
    """
    lon_rad = np.deg2rad(lon)
    radius = 90.0 - lat
    x = radius * np.sin(lon_rad)
    y = -radius * np.cos(lon_rad)
    return x, y


def points_inside_polygon(lon, lat, polygon_lon, polygon_lat):
    """
    Return an xarray boolean mask indicating whether each grid
    point is inside the polygon.
    """
    lat_2d, lon_2d = xr.broadcast(lat, lon)

    grid_x, grid_y = polar_xy(np.asarray(lon_2d),np.asarray(lat_2d))
    polygon_x, polygon_y = polar_xy(np.asarray(polygon_lon),np.asarray(polygon_lat))
    vertices = np.column_stack([polygon_x, polygon_y])

    # Explicitly close the polygon
    vertices = np.vstack([vertices, vertices[0]])
    polygon = mpath.Path(vertices)
    points = np.column_stack([grid_x.ravel(),grid_y.ravel()])
    inside = polygon.contains_points(points).reshape(grid_x.shape)
    return xr.DataArray(inside,dims=lat_2d.dims,coords=lat_2d.coords)


def make_arctic_basin_masks(lon,lat,base_mask=None,lat_min=65.0,):
    """
    Construct approximate Eurasian and Amerasian Basin masks
    using the Lomonosov Ridge dividing line.
    """

    # Eurasian side:
    # close the ridge through the 65°N boundary via
    # Greenland -> Atlantic/Barents -> Siberia
    eurasian_arc_lon = np.linspace(-60, 140, 361)
    eurasian_arc_lat = np.full_like(eurasian_arc_lon,lat_min,dtype=float)

    eurasian_polygon_lon = np.concatenate([ridge_lon,[-60],eurasian_arc_lon,[140]])
    eurasian_polygon_lat = np.concatenate([ridge_lat,[lat_min],eurasian_arc_lat,[lat_min]])

    # Amerasian side:
    # use the other part of the 65°N boundary via
    # Canada/Alaska/Chukchi -> Siberia
    amerasian_arc_lon = np.linspace(-60, -220, 361)
    amerasian_arc_lat = np.full_like(amerasian_arc_lon,lat_min,dtype=float)
    amerasian_polygon_lon = np.concatenate([ridge_lon,[-60],amerasian_arc_lon,[140]])
    amerasian_polygon_lat = np.concatenate([ridge_lat,[lat_min],amerasian_arc_lat,[lat_min]])

    eurasian_mask = points_inside_polygon(lon,lat,eurasian_polygon_lon,eurasian_polygon_lat)
    amerasian_mask = points_inside_polygon(lon,lat,amerasian_polygon_lon,amerasian_polygon_lat)

    if base_mask is not None:
        eurasian_mask = eurasian_mask & base_mask
        amerasian_mask = amerasian_mask & base_mask

    return (
        eurasian_mask.rename("eurasian_basin_mask"),
        amerasian_mask.rename("amerasian_basin_mask"),
    )

In [ ]:
with xr.open_dataset(grid_file) as grid:

    deep_arctic_mask = ((grid["pmask"] > 0)& (grid["plat"] >= 65)& (grid["pdepth"] > 500))
    eurasian_mask, amerasian_mask = make_arctic_basin_masks(lon=grid["plon"],lat=grid["plat"],base_mask=deep_arctic_mask,lat_min=65)
    basin_weights = {"eurasian": (grid["parea"].where(eurasian_mask, 0).fillna(0).load()),
        "amerasian": (grid["parea"].where(amerasian_mask, 0).fillna(0).load())}

profiles = {"eurasian": {"temp": [],"salt": []},"amerasian": {"temp": [],"salt": []}}


for file in files:
    with xr.open_dataset(file,decode_times=time_coder,) as ds:
        for basin, weights in basin_weights.items():
            temp_one = (ds["templvl"].sel(depth=slice(0, 4000)).weighted(weights).mean(("y", "x"), skipna=True).load())
            salt_one = (ds["salnlvl"].sel(depth=slice(0, 4000)).weighted(weights).mean(("y", "x"), skipna=True).load())
            profiles[basin]["temp"].append(temp_one)
            profiles[basin]["salt"].append(salt_one)
temp_profile_eurasian = (xr.concat(profiles["eurasian"]["temp"],dim="time").mean("time", skipna=True))
salt_profile_eurasian = (xr.concat(profiles["eurasian"]["salt"],dim="time").mean("time", skipna=True))
temp_profile_amerasian = (xr.concat(profiles["amerasian"]["temp"],dim="time").mean("time", skipna=True))
salt_profile_amerasian = (xr.concat(profiles["amerasian"]["salt"],dim="time").mean("time", skipna=True))

phc_temp_profile_eurasian = (phc_temp_3d_on_model.weighted(basin_weights["eurasian"]).mean(("y", "x"), skipna=True))
phc_salt_profile_eurasian = (phc_salt_3d_on_model.weighted(basin_weights["eurasian"]).mean(("y", "x"), skipna=True))
phc_temp_profile_amerasian = (phc_temp_3d_on_model.weighted(basin_weights["amerasian"]).mean(("y", "x"), skipna=True))
phc_salt_profile_amerasian = (phc_salt_3d_on_model.weighted(basin_weights["amerasian"]).mean(("y", "x"), skipna=True))

In [ ]:
# These are exactly the grid cells used in your calculation
eur = basin_weights["eurasian"] > 0
amer = basin_weights["amerasian"] > 0
region = xr.where(eur,1.0,xr.where(amer, 2.0, np.nan),)
region.attrs["units"] = "Basin"
cmap = mcolors.ListedColormap(["tab:blue","tab:orange"])
norm = mcolors.BoundaryNorm([0.5, 1.5, 2.5],cmap.N)

with xr.open_dataset(grid_file) as grid:
    plon = grid["plon"].load()
    plat = grid["plat"].load()
    deep_arctic = ((grid["pmask"] > 0)& (grid["plat"] >= 65)& (grid["pdepth"] > 500)).load()

fig, ax = plot_map2d(lon=plon,lat=plat,var=region,proj="NorthPolarStereo",norm=norm,cmap=cmap)
ax.set_extent([-180, 180, 65, 90],crs=ccrs.PlateCarree(),)
ax.plot(ridge_lon,ridge_lat,color="magenta",linewidth=2,transform=ccrs.Geodetic(),label="Lomonosov Ridge approximation",zorder=5,)
ax.set_title("Deep-Arctic basin masks",fontsize=14,)
ax.legend(loc="lower left")
cbar_ax = fig.axes[-1]
cbar_ax.set_yticks([1, 2])
cbar_ax.set_yticklabels(["Eurasian Basin","Amerasian Basin"])
cbar_ax.set_ylabel("Basin")
theta = np.linspace(0, 2 * np.pi, 200)

circle = mpath.Path(np.column_stack([0.5 + 0.5 * np.cos(theta),0.5 + 0.5 * np.sin(theta)]))
ax.set_boundary(circle,transform=ax.transAxes)
plt.show()

In [ ]:
profiles = {
    "Eurasian Basin": {"Temperature": {"model": temp_profile_eurasian,"phc": phc_temp_profile_eurasian},"Salinity": {"model": salt_profile_eurasian,"phc": phc_salt_profile_eurasian}},
    "Amerasian Basin": {"Temperature": {"model": temp_profile_amerasian,"phc": phc_temp_profile_amerasian},"Salinity": {"model": salt_profile_amerasian,"phc": phc_salt_profile_amerasian}}}
variable_settings = {"Temperature": {"xlabel": "Temperature [°C]","title": "temperature profile"},
    "Salinity": {"xlabel": r"Salinity [g kg$^{-1}$]","title": "salinity profile"}}
variable_colors = {"Temperature": "tab:red","Salinity": "tab:blue"}
for basin, basin_profiles in profiles.items():
    for variable, variable_profiles in basin_profiles.items():
        model_profile = variable_profiles["model"]
        phc_profile = variable_profiles["phc"]
        settings = variable_settings[variable]
        model_color = variable_colors[variable]
        fig, ax = plt.subplots(figsize=(6, 6.5))
        # NorESM3
        ax.plot(model_profile.values,model_profile["depth"].values / 1000,color= model_color,linewidth=2.2,label="NorESM3")
        # PHC3.0
        ax.plot(phc_profile.values,phc_profile["depth"].values / 1000,color= 'black',linestyle="-",linewidth=2,label="PHC3.0",)
        ax.set_xlabel(settings["xlabel"],fontsize=12,)
        ax.set_ylabel("Depth [km]",fontsize=12,)
        ax.set_title(f"{basin} {settings['title']}",fontsize=14,)
        ax.set_ylim(4, 0)
        ax.grid(True, alpha=0.3)
        ax.legend()
        fig.suptitle(f"{caseid}\nYears {year_start}–{year_end}",fontsize=12,y=0.98,)
        fig.tight_layout(rect=[0, 0, 1, 0.95])
        plt.show()

## Climatology maps
1. surface T & S
2. 400m T & S
3. 1000m T & S
4. 2000m T & S

In [ ]:
# Climatology maps
depths = [0,400,1000,2000]
plat, plon, pclat, pclon , pmask = get_grid.load(grid_file)

temp_model_arctic={}
salt_model_arctic={}
phc_temp_on_model={}
phc_salt_on_model={}
temp_bias={}
salt_bias={}
temp_levels={}
salt_levels={}
temp_bias_levels={}
salt_bias_levels={}
for depth in depths:
    temp_model=mean_at_depth("templvl",files,depth)
    salt_model=mean_at_depth("salnlvl",files,depth)
    ydim, xdim = temp_model.dims[-2:]
    # Arctic
    arctic = xr.DataArray((np.asarray(pmask) > 0) & (np.asarray(plat) >= 60),dims=(ydim, xdim))
    temp_model_arctic[depth] = temp_model.where(arctic)
    salt_model_arctic[depth] = salt_model.where(arctic)
    # interp
    with xr.open_dataset(phc_file, decode_times=time_coder) as ds:
        phc_temp = ds["temp"].interp(depth=depth).load()
        phc_salt = ds["salt"].interp(depth=depth).load()
    # PHC
    phc_grid = xr.Dataset(coords={"lon": phc_temp["lon"],"lat": phc_temp["lat"],})
    # PHC → model grid
    regridder = xe.Regridder(phc_grid,target_grid,method="bilinear",periodic=True,)
    # skipnan
    phc_temp_on_model[depth] = regridder(phc_temp,skipna=True,na_thres=1.0,).where(arctic)
    phc_salt_on_model[depth] = regridder(phc_salt,skipna=True,na_thres=1.0,).where(arctic)
    # Bias
    temp_bias[depth] = temp_model_arctic[depth]- phc_temp_on_model[depth]
    salt_bias[depth] = salt_model_arctic[depth]- phc_salt_on_model[depth]
    temp_levels[depth] = shared_quantile_levels([temp_model_arctic[depth]],nlevels=21)
    salt_levels[depth] = shared_quantile_levels([salt_model_arctic[depth]],nlevels=21)
    temp_bias_levels[depth] = symmetric_quantile_levels(temp_bias[depth],nlevels=21)
    salt_bias_levels[depth] =symmetric_quantile_levels(salt_bias[depth],nlevels=21)


def plot_arctic_climatology(variable, depth):

    if variable in ("temperature", "temp"):
        data = temp_model_arctic[depth]
        levels = temp_levels[depth]
        cmap = mpl.colormaps["plasma"]
        units = "°C"
        variable_title = "Temperature"

    elif variable in ("salinity", "salt"):
        data = salt_model_arctic[depth]
        levels = salt_levels[depth]
        cmap = mpl.colormaps["viridis"]
        units = "psu"
        variable_title = "Salinity"

    else:
        raise ValueError(
            "variable must be 'temperature' or 'salinity'"
        )

    norm = mpl.colors.BoundaryNorm(levels,ncolors=cmap.N,clip=False)
    data = data.copy(deep=False)
    data.attrs = dict(data.attrs)
    data.attrs.setdefault("units", units)

    fig, ax = plot_map2d(lon=plon,lat=plat,var=data,proj="NorthPolarStereo",norm=norm,cmap=cmap)
    ax.set_extent([-180, 180, 60, 90],crs=ccrs.PlateCarree())
    # Set a circular map boundary
    theta = np.linspace(0, 2 * np.pi, 200)
    circle = mpath.Path(np.column_stack([ 0.5 + 0.5 * np.cos(theta),0.5 + 0.5 * np.sin(theta)]))
    ax.set_boundary(circle,transform=ax.transAxes)
    ax.set_title(f"{caseid}\n"
            f"Years {year_start}–{year_end}, depth = {depth} m\n"
            f"model Arctic Ocean {variable}")
    fig.tight_layout()
    plt.show()

    return fig, ax

In [ ]:
for d in depths:
    plot_arctic_climatology('temperature', d)
    plot_arctic_climatology('salinity', d)

## Bias maps
model_result - PHC3.0_results
1. surface
2. 400m
3. 1000m
4. 2000m

In [ ]:
def plot_arctic_bias(variable, depth):

    if variable in ("temperature", "temp"):
        data = temp_bias[depth]
        levels = temp_bias_levels[depth]
        cmap = mpl.colormaps["RdBu_r"]
        units = "°C"
        variable_title = "Temperature bias"

    elif variable in ("salinity", "salt"):
        data = salt_bias[depth]
        levels = salt_bias_levels[depth]
        cmap = mpl.colormaps["RdBu_r"]
        units = "psu"
        variable_title = "Salinity bias"

    else:
        raise ValueError(
            "variable must be 'temperature' or 'salinity'"
        )

    norm = mpl.colors.BoundaryNorm(levels,ncolors=cmap.N,clip=False)
    data = data.copy(deep=False)
    data.attrs = dict(data.attrs)
    data.attrs.setdefault("units", units)

    fig, ax = plot_map2d(lon=plon,lat=plat,var=data,proj="NorthPolarStereo",norm=norm,cmap=cmap)
    ax.set_extent([-180, 180, 60, 90],crs=ccrs.PlateCarree())
    # Set a circular map boundary
    theta = np.linspace(0, 2 * np.pi, 200)
    circle = mpath.Path(np.column_stack([ 0.5 + 0.5 * np.cos(theta),0.5 + 0.5 * np.sin(theta)]))
    ax.set_boundary(circle,transform=ax.transAxes)
    ax.set_title(f"{caseid}\n"
            f"Years {year_start}–{year_end}, depth = {depth} m\n"
            f"{variable} bias")
    fig.tight_layout()
    plt.show()
    return fig, ax
    

In [ ]:
for d in depths:
    plot_arctic_bias('temperature', d)
    plot_arctic_bias('salinity', d)

## Transects
Greenland to Svalbard profile
1. temperature section
2. salinity section
3. normal velocity section
4. volume transport
5. heat transport
6. freshwater transport

### basins

### transects of T & S

In [ ]:
target_lat = 78.83
target_lons = np.linspace(-20, 15, 36)
max_depth = 1000

def spherical_xyz(lon, lat):
    lon = np.deg2rad(np.asarray(lon))
    lat = np.deg2rad(np.asarray(lat))
    return np.column_stack([
        np.cos(lat) * np.cos(lon),
        np.cos(lat) * np.sin(lon),
        np.sin(lat),
    ])


with xr.open_dataset(grid_file) as grid_ds:
    plon = grid_ds["plon"].load()
    plat = grid_ds["plat"].load()
    pmask = grid_ds["pmask"].load()
    pdepth = grid_ds["pdepth"].load()

ydim, xdim = plat.dims
model_lon = np.asarray(plon)
model_lat = np.asarray(plat)
model_mask = np.asarray(pmask) > 0
model_bottom = np.asarray(pdepth)

model_lon_180 = (model_lon + 180) % 360 - 180
valid_coordinates = (np.isfinite(model_lon)& np.isfinite(model_lat))
valid_y, valid_x = np.where(valid_coordinates)
tree = cKDTree(spherical_xyz(model_lon[valid_coordinates],model_lat[valid_coordinates]))
target_xyz = spherical_xyz(target_lons,np.full_like(target_lons,target_lat,dtype=float),)
# Find nearest BLOM grid cell
distance, nearest = tree.query(target_xyz)
section_y = valid_y[nearest]
section_x = valid_x[nearest]

section_actual_lon = model_lon[section_y,section_x]
section_actual_lat = model_lat[section_y,section_x]
## selected the nearest grid in grid_file

section_wet = model_mask[section_y,section_x] #bool
section_bottom_raw = model_bottom[section_y,section_x]

# Convert selected longitude into -180 to 180
section_actual_lon_180 = (section_actual_lon + 180) % 360 - 180
section_bottom = np.where(section_wet,section_bottom_raw,0.0)
y_index = xr.DataArray(section_y,dims="section")
x_index = xr.DataArray(section_x,dims="section")

temp_sections = []
salt_sections = []
for i, file in enumerate(files, 1):
    with xr.open_dataset(file,decode_times=time_coder) as ds:
        temp_one = (ds["templvl"].sel(depth=slice(0, max_depth)).isel({ydim: y_index,xdim: x_index}).mean("time").load())
        salt_one = (ds["salnlvl"].sel(depth=slice(0, max_depth)).isel({ ydim: y_index,xdim: x_index}).mean("time").load())
    temp_sections.append(temp_one)
    salt_sections.append(salt_one)
temp_section = (xr.concat(temp_sections,dim="year").mean("year"))
salt_section = (xr.concat(salt_sections,dim="year").mean("year"))

# Assign target longitude as the section coordinate
temp_section = temp_section.assign_coords(longitude=("section",target_lons))
salt_section = salt_section.assign_coords(longitude=("section",target_lons))

# Apply ocean and bathymetry masks
wet_section = xr.DataArray(section_wet,dims="section",coords={"section": temp_section["section"],"longitude": ("section",target_lons),})
bottom = xr.DataArray(section_bottom,dims="section",coords={"section": temp_section["section"],"longitude": ("section",target_lons),})
section_valid = (wet_section& (temp_section["depth"] <= bottom))
temp_section = temp_section.where(section_valid)
salt_section = salt_section.where(section_valid)


In [ ]:
projection = ccrs.NorthPolarStereo(central_longitude=0)
data_crs = ccrs.PlateCarree()
fig, ax = plt.subplots(figsize=(9, 8),subplot_kw={ "projection": projection,})
# Ocean bathymetry
bathymetry = np.where(model_mask,model_bottom,np.nan)
depth_levels = [0,200,500,1000,2000,3000,4000,5000]
depth_norm = mcolors.BoundaryNorm(depth_levels,ncolors=plt.colormaps["Blues"].N)
bathymetry_plot = ax.pcolormesh(model_lon_180,model_lat,bathymetry,transform=data_crs,cmap="Blues",norm=depth_norm,shading="auto",zorder=1)

# Land
ax.add_feature( cfeature.LAND,facecolor="0.7",edgecolor="black",linewidth=0.5,zorder=3)
ax.coastlines(resolution="50m",linewidth=0.6,zorder=4)

# Ideal target section
ax.plot(target_lons,np.full_like(target_lons,target_lat,dtype=float,),color="red",linewidth=2,transform=data_crs,label=f"Target section: {target_lat:.2f}°N",zorder=6)
# Connect the actual selected grid-cell centres
ax.plot(section_actual_lon_180,section_actual_lat,color="black",linestyle="--",linewidth=0.8,transform=data_crs,label="Actual selected grid path",zorder=7)
# Selected ocean cells
ax.scatter(section_actual_lon_180[section_wet],section_actual_lat[section_wet],s=28,color="black",edgecolor="white",linewidth=0.5,transform=data_crs,label="Selected ocean cells",zorder=8)

# Selected land cells
ax.scatter(section_actual_lon_180[~section_wet],section_actual_lat[~section_wet],s=36,marker="x",color="magenta",linewidth=1.3,transform=data_crs,label="Selected land cells",zorder=9)
# Regional map extent
ax.set_extent([-30, 25, 74, 83],crs=data_crs,)
gridlines = ax.gridlines(crs=data_crs,draw_labels=True,x_inline=False,y_inline=False,xlocs=np.arange(-30, 31, 10),ylocs=np.arange(74, 84, 2),linewidth=0.4,color="0.4",alpha=0.6,linestyle="--",)
gridlines.top_labels = False
gridlines.right_labels = False
ax.set_title("Location of the Fram Strait transect\n"f"{caseid}",fontsize=11,)
fig.colorbar(bathymetry_plot,ax=ax,label="Ocean depth [m]",shrink=0.78,pad=0.05)

ax.legend(loc="lower left",fontsize=8,framealpha=0.9)
plt.show()

In [ ]:
#add T and S 
temperature_contours = [-1.5, -1.0, 0.0, 1.0, 2.0, 2.5]
salinity_contours = [32.0, 33.0, 34.0, 34.5, 34.7, 34.9, 35.0]

fig, axes = plt.subplots(2,1,figsize=(11, 9),sharex=True)
# Temperature
temp_plot = axes[0].pcolormesh(temp_section["longitude"],temp_section["depth"],temp_section,cmap="plasma",shading="auto")
salt_lines_on_temp = axes[0].contour(salt_section["longitude"],salt_section["depth"],salt_section,levels=salinity_contours,colors="black",linewidths=0.7,)
axes[0].clabel(salt_lines_on_temp,inline=True,fontsize=8,fmt=lambda value:f"S={value:.1f}")
axes[0].fill_between(target_lons,section_bottom,max_depth,color="0.4")
axes[0].set_ylim(max_depth , 0)
axes[0].set_ylabel("Depth[m]")
axes[0].set_title("Temperature")
axes[0].grid(alpha=0.2)
fig.colorbar(temp_plot,ax=axes[0],label="Temperature [°C]")

# Salinity
salt_plot = axes[1].pcolormesh(salt_section["longitude"],salt_section["depth"],salt_section,cmap="viridis",shading="auto")
temp_lines_on_salt = axes[1].contour(temp_section["longitude"],temp_section["depth"],temp_section,levels=temperature_contours,colors="black",linewidths=0.7,)
axes[1].clabel(temp_lines_on_salt,inline=True,fontsize=8,fmt=lambda value:f"T={value:.1f}")
axes[1].fill_between(target_lons,section_bottom,max_depth,color="0.4")
axes[1].set_ylim(max_depth,0)
axes[1].set_xlabel("Longitude [°E]")
axes[1].set_ylabel("Depth [m]")
axes[1].set_title("Salinity")
axes[1].grid(alpha=0.2)

fig.colorbar(salt_plot,ax=axes[1],label="Salinity [g kg$^{-1}$]")
fig.suptitle(
    f"{caseid}\n"
    f"Fram Strait section near {target_lat:.2f}°N, "
    f"years {year_start}–{year_end}",
    fontsize=11,
)

fig.subplots_adjust(top=0.89,hspace=0.25)
plt.show()